# Load SL Model

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
import numpy as np
from torch_geometric.data import Data, InMemoryDataset
import torch
from torch_cluster import knn_graph
from torch.nn import Sequential, Linear, ReLU
import torch.nn.functional as F
from torch_geometric.nn import global_max_pool, global_mean_pool, MessagePassing
from torch_geometric.nn.inits import reset
from torch import Tensor

In [2]:
class PointMPLayer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        # Message passing with "max" aggregation.
        super().__init__(aggr='max')

        # Initialization of the MLP:
        # Here, the number of input features correspond to the hidden node
        # dimensionality plus point dimensionality (=3).
        self.mlp = Sequential(Linear(in_channels + 3, out_channels),
                              ReLU(),
                              Linear(out_channels, out_channels))

    def forward(self, h, pos, edge_index):
        # Start propagating messages.
        return self.propagate(edge_index, h=h, pos=pos)

    def message(self, h_j, pos_j, pos_i):
        # h_j defines the features of neighboring nodes as shape [num_edges, in_channels]
        # pos_j defines the position of neighboring nodes as shape [num_edges, 3]
        # pos_i defines the position of central nodes as shape [num_edges, 3]

        input = pos_j - pos_i  # Compute spatial relation.

        if h_j is not None:
            # In the first layer, we may not have any hidden node features,
            # so we only combine them in case they are present.
            input = torch.cat([h_j, input], dim=-1)

        return self.mlp(input)  # Apply our final MLP.

In [3]:
class Encoder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = PointMPLayer(3, 64)
        self.conv2 = PointMPLayer(64, 128)
        self.conv3 = PointMPLayer(128, 256)
        self.linear1 = Linear(128, 1)
        self.linear2 = Linear(256, 512)
        self.linear3 = Linear(256, 1)
        self.linear4 = Linear(256, 128)

    def forward(self, pos, edge_index, batch):
        # Compute the kNN graph:
        # Here, we need to pass the batch vector to the function call in order
        # to prevent creating edges between points of different examples.
        # We also add `loop=True` which will add self-loops to the graph in
        # order to preserve central point information.
        #edge_index = knn_graph(pos, k=16, batch=batch, loop=True)

        # Start bipartite message passing.
        h = self.conv1(h=pos, pos=pos, edge_index=edge_index)
        h = h.relu()
        h = self.conv2(h=h, pos=pos, edge_index=edge_index)
        h = h.relu()
        h = self.conv3(h=h, pos=pos, edge_index=edge_index)
        h = h.relu()

        # Global Pooling.
        h_global = global_mean_pool(self.linear3(h), batch)  # [num_examples, hidden_channels]
        h_local = self.linear1(self.linear4(h).relu())
        h_latent = self.linear2(h)
        z = global_mean_pool(h_latent, batch)
        
        return h_global, h_local, h_latent, z, h

In [4]:
class Decoder(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = PointMPLayer(3, 32)
        self.conv4 = PointMPLayer(32, 3)
        self.linear3 = Linear(1, 8759)
        self.linear4 = Linear(1, 8759)
        self.linear5 = Linear(256, 128)
        self.linear6 = Linear(128, 32)
        self.linear7 = Linear(32, 3)
        self.linear8 = Linear(512, 256)

    def forward(self, z):

        #z_topo = self.linear3(z).relu()
        #adj = torch.sigmoid(torch.matmul(z_topo, z_topo.t()))
        #edge_index_d = adj.nonzero().t().contiguous()
        #z_pos = self.linear4(z).relu().reshape(-1,1)
        #z_pos = self.linear5(z_pos)
        z_pos = self.linear5(self.linear8(z).relu()).relu()
        z_pos = self.linear6(z_pos).relu()
        z_pos = self.linear7(z_pos)
        ###edge_index_d = knn_graph(z_pos, k=3, batch=batch, loop=True)
        
        # Start bipartite message passing.
        #h_d = self.conv3(h=z_pos, pos=z_pos, edge_index=edge_index_d)
        #h_d = h_d.relu()
        #h_d = self.conv4(h=h_d, pos=z_pos, edge_index=edge_index_d)
    
        return z_pos

In [5]:
class AE(torch.nn.Module):
    
    def __init__(self, encoder: torch.nn.Module, decoder: torch.nn.Module):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        AE.reset_parameters(self)

    def reset_parameters(self):
        r"""Resets all learnable parameters of the module."""
        reset(self.encoder)
        reset(self.decoder)


    def forward(self, *args, **kwargs) -> Tensor:  # pragma: no cover
        r"""Alias for :meth:`encode`."""
        return self.encoder(*args, **kwargs)
    def encode(self, *args, **kwargs) -> Tensor:
        r"""Runs the encoder and computes node-wise latent variables."""
        return self.encoder(*args, **kwargs)


    def decode(self, *args, **kwargs) -> Tensor:
        r"""Runs the decoder and computes edge probabilities."""
        return self.decoder(*args, **kwargs)

In [6]:
model = AE(Encoder(),Decoder()).to('cuda')

In [7]:
model.load_state_dict(torch.load('./SL.pth'))

<All keys matched successfully>

# Extrapolation on Ellipsoidal Data

In [8]:
folder_list = os.listdir('./ellipsoid_data')

In [9]:
data = []
count2 = 0
for folder in folder_list:
    thickness = folder.split('_')[2]
    u = folder.split('_')[3]
    folder_path = os.path.join('./ellipsoid_data',folder)
    num = len(os.listdir(folder_path))
    for i in range(num):
        count = i*500
        file_name = thickness+'_'+u+'_'+str(count)+'.txt'
        file_name_path = os.path.join(folder_path,file_name)
        data1 = np.loadtxt(file_name_path)
        if i < num/2:
            e_i = knn_graph(torch.from_numpy(data1[:,:3]), k=3, loop=True).to('cuda')
        else:
            e_i = knn_graph(torch.from_numpy(data1[:,:3]), k=3, loop=True).to('cuda')
        data.append(Data(x=torch.from_numpy(data1[:,:3]).to('cuda',dtype=torch.float), edge_index = e_i))
        print(count2)
        print(file_name_path)
        count2+=1

0
./ellipsoid_data/res_e_0.042_u1/0.042_u1_0.txt
1
./ellipsoid_data/res_e_0.042_u1/0.042_u1_500.txt
2
./ellipsoid_data/res_e_0.042_u1/0.042_u1_1000.txt
3
./ellipsoid_data/res_e_0.042_u1/0.042_u1_1500.txt
4
./ellipsoid_data/res_e_0.042_u1/0.042_u1_2000.txt
5
./ellipsoid_data/res_e_0.042_u1/0.042_u1_2500.txt
6
./ellipsoid_data/res_e_0.042_u1/0.042_u1_3000.txt
7
./ellipsoid_data/res_e_0.042_u1/0.042_u1_3500.txt
8
./ellipsoid_data/res_e_0.042_u1/0.042_u1_4000.txt
9
./ellipsoid_data/res_e_0.042_u1/0.042_u1_4500.txt
10
./ellipsoid_data/res_e_0.042_u1/0.042_u1_5000.txt
11
./ellipsoid_data/res_e_0.042_u1/0.042_u1_5500.txt
12
./ellipsoid_data/res_e_0.042_u1/0.042_u1_6000.txt
13
./ellipsoid_data/res_e_0.042_u1/0.042_u1_6500.txt
14
./ellipsoid_data/res_e_0.042_u1/0.042_u1_7000.txt
15
./ellipsoid_data/res_e_0.042_u1/0.042_u1_7500.txt
16
./ellipsoid_data/res_e_0.042_u1/0.042_u1_8000.txt
17
./ellipsoid_data/res_e_0.042_u1/0.042_u1_8500.txt
18
./ellipsoid_data/res_e_0.042_u1/0.042_u1_9000.txt
19
./el

In [10]:
lat = []
for i in range(64):
    h_global, h_local, h_latent, z, h = model.encode(data[i].x,data[i].edge_index, data[i].batch)  # Forward pass.
    h_d = model.decode(h_latent)
    lat.append(h_local.mean().detach().cpu().numpy())
np.savetxt('./SL_ellipsoid_pred.txt', lat)

In [11]:
lat

[array(2.4628906, dtype=float32),
 array(2.5707638, dtype=float32),
 array(2.6824946, dtype=float32),
 array(2.8074899, dtype=float32),
 array(2.9150891, dtype=float32),
 array(3.001165, dtype=float32),
 array(3.0627403, dtype=float32),
 array(3.1389754, dtype=float32),
 array(3.1896625, dtype=float32),
 array(3.2747288, dtype=float32),
 array(3.3404043, dtype=float32),
 array(3.426912, dtype=float32),
 array(3.5148368, dtype=float32),
 array(3.6179633, dtype=float32),
 array(3.737842, dtype=float32),
 array(3.8567307, dtype=float32),
 array(3.9909754, dtype=float32),
 array(4.1424947, dtype=float32),
 array(4.2974925, dtype=float32),
 array(4.4624295, dtype=float32),
 array(4.6334777, dtype=float32),
 array(4.812807, dtype=float32),
 array(5.0131936, dtype=float32),
 array(5.2137437, dtype=float32),
 array(5.414083, dtype=float32),
 array(5.631554, dtype=float32),
 array(5.880709, dtype=float32),
 array(6.1810603, dtype=float32),
 array(6.5763984, dtype=float32),
 array(7.139242, dtyp

# Extrapolation on Brain Data

In [12]:
datab = []
folder_list2 = os.listdir('./brain_data')
folder_list2.sort()
for folder in folder_list2:
        file_name_path = os.path.join('./brain_data',folder)
        data1 = np.loadtxt(file_name_path)
        e_i = knn_graph(torch.from_numpy(data1[:,:3]), k=12, loop=True).to('cuda')
        datab.append(Data(x=torch.from_numpy(data1[:,:3]).to('cuda',dtype=torch.float), edge_index = e_i))
        print(file_name_path)

./brain_data/fetal.week21.right.pial.surf.txt
./brain_data/fetal.week22.right.pial.surf.txt
./brain_data/fetal.week23.right.pial.surf.txt
./brain_data/fetal.week24.right.pial.surf.txt
./brain_data/fetal.week25.right.pial.surf.txt
./brain_data/fetal.week26.right.pial.surf.txt
./brain_data/fetal.week27.right.pial.surf.txt
./brain_data/fetal.week28.right.pial.surf.txt
./brain_data/fetal.week29.right.pial.surf.txt
./brain_data/fetal.week30.right.pial.surf.txt
./brain_data/fetal.week31.right.pial.surf.txt
./brain_data/fetal.week32.right.pial.surf.txt
./brain_data/fetal.week33.right.pial.surf.txt
./brain_data/fetal.week34.right.pial.surf.txt
./brain_data/fetal.week35.right.pial.surf.txt
./brain_data/fetal.week36.right.pial.surf.txt


In [13]:
latb = []
for i in range(16):
    h_global, h_local, h_latent, z, h = model.encode(datab[i].x, datab[i].edge_index, datab[i].batch)  # Forward pass.
    h_d = model.decode(h_latent)
    latb.append(h_local.mean().detach().cpu().numpy())
np.savetxt('./SL_brain_pred.txt', latb)

In [14]:
latb

[array(3.0229714, dtype=float32),
 array(3.0251334, dtype=float32),
 array(3.028858, dtype=float32),
 array(3.044674, dtype=float32),
 array(3.0773547, dtype=float32),
 array(3.1223207, dtype=float32),
 array(3.226033, dtype=float32),
 array(3.374885, dtype=float32),
 array(3.560137, dtype=float32),
 array(3.8904033, dtype=float32),
 array(4.319637, dtype=float32),
 array(4.7222276, dtype=float32),
 array(5.1355295, dtype=float32),
 array(5.607682, dtype=float32),
 array(5.9653134, dtype=float32),
 array(6.259774, dtype=float32)]

# Analysis of Neural Patterns

In [15]:
datab = []
folder_list2 = os.listdir('./brain_data')
folder_list2.sort()
for folder in folder_list2:
        file_name_path = os.path.join('./brain_data',folder)
        data1 = np.loadtxt(file_name_path)
        e_i = knn_graph(torch.from_numpy(data1[:,:3]), k=12, loop=True).to('cuda')
        datab.append(Data(x=torch.from_numpy(data1[:,:3]).to('cuda',dtype=torch.float), edge_index = e_i))
        print(file_name_path)

./brain_data/fetal.week21.right.pial.surf.txt
./brain_data/fetal.week22.right.pial.surf.txt
./brain_data/fetal.week23.right.pial.surf.txt
./brain_data/fetal.week24.right.pial.surf.txt
./brain_data/fetal.week25.right.pial.surf.txt
./brain_data/fetal.week26.right.pial.surf.txt
./brain_data/fetal.week27.right.pial.surf.txt
./brain_data/fetal.week28.right.pial.surf.txt
./brain_data/fetal.week29.right.pial.surf.txt
./brain_data/fetal.week30.right.pial.surf.txt
./brain_data/fetal.week31.right.pial.surf.txt
./brain_data/fetal.week32.right.pial.surf.txt
./brain_data/fetal.week33.right.pial.surf.txt
./brain_data/fetal.week34.right.pial.surf.txt
./brain_data/fetal.week35.right.pial.surf.txt
./brain_data/fetal.week36.right.pial.surf.txt


In [16]:
latb = []
for i in range(16):
    h_global, h_local, h_latent, z, h = model.encode(datab[i].x, datab[i].edge_index, datab[i].batch)  # Forward pass.
    h_d = model.decode(h_latent)
    latb.append(h.detach().cpu().numpy())

In [17]:
lat_arr = np.array(latb).reshape(-1,256)
lat_arr_mean = lat_arr.mean(axis = 0)
lat_arr_mean.shape

(256,)

In [18]:
np.savetxt('SL_sp_br_patt.txt',lat_arr_mean)

In [19]:
folder_list = os.listdir('./sphere_data_inference')
data = []
count2 = 0
for folder in folder_list:
    thickness = folder.split('_')[2]
    u = folder.split('_')[3]
    folder_path = os.path.join('./sphere_data_inference',folder)
    num = len(os.listdir(folder_path))
    lat = []
    for i in range(num):
        count = i*500
        file_name = thickness+'_'+u+'_'+str(count)+'.txt'
        file_name_path = os.path.join(folder_path,file_name)
        data1 = np.loadtxt(file_name_path)
        e_i = knn_graph(torch.from_numpy(data1[:,:3]), k=3, loop=True).to('cuda')
        data.append(Data(x=torch.from_numpy(data1[:,:3]).to('cuda',dtype=torch.float), edge_index = e_i, curv = torch.from_numpy(data1[:,6]).to('cuda',dtype=torch.float)))
        print(count2)
        count2+=1
        h_global, h_local, h_latent, z, h = model.encode(torch.from_numpy(data1[:,:3]).to('cuda',dtype=torch.float), e_i, data[0].batch)  # Forward pass.
        lat.append(h.detach().cpu().numpy())

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178


In [20]:
lat_arr = np.array(lat).reshape(-1,256)

In [21]:
lat_arr.shape

(630648, 256)

In [22]:
lat_arr_mean = lat_arr.mean(axis = 0)

In [23]:
np.savetxt('SL_sp_sp_patt.txt',lat_arr_mean)